<a href="https://colab.research.google.com/github/TheAlishbahWaheed/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TheAlishbahWaheed/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

## My ranked action queue

**Score:** predicted probability of decline (`oof_model_proba`), from the same Random Forest
and leakage-safe feature set trained in `w05_model.ipynb`, scored with 5-fold **client-grouped
out-of-fold (OOF)** predictions — so every one of the 30,000 rows gets a genuinely out-of-sample
score, not just the 25% that sat in the Week-5/Week-6 test split. The audited, reported
performance number stays the single grouped split from `w06_validation_audit.ipynb` (ROC-AUC
0.603, precision@50 0.540 — re-checked below, matches). The OOF scores exist only to cover the
full population for this playbook; their own honest AUC is 0.673, with a fold range of
0.602–0.735 across only 32 clients — expect this kind of spread, it's the client count, not
model instability.

**Reason codes** are transparent, threshold-based flags a human can read without a model:
`model_decline_risk` (OOF probability ≥ 0.65), `stale_and_visible`, `high_demand_low_ctr`,
`low_engagement`, `thin_but_visible`, `already_winning`, `no_real_demand`. A row can carry more
than one.

**Archetypes** are named groups built by crossing those same flags (priority order: protect
first, then each decline-risk combination, then no-flag). This is a rule-based cross, **not** a
statistical cluster — no k-means or PCA was run to produce these — so I'm calling it a readable
label for a readable rule, not "discovered" or "semantic" segmentation.

| Archetype | n | Observed decline rate | Mapped action |
|---|---:|---:|---|
| Stale Workhorse | 2,254 | 0.687 | `refresh_priority` |
| Visible But Under-Clicked | 2,582 | 0.686 | `rewrite_title_and_meta` |
| Disengaging Reader | 221 | 0.692 | `review_engagement_and_layout` |
| General Decline Risk (flagged, no other pattern) | 3,555 | 0.692 | `refresh_review` |
| Thin But Wanted | 1 | 1.000 (n=1 — noted, not trusted) | `expand_and_refresh` |
| Champion (protect) | 2,647 | 0.471 | `protect_do_not_touch` |
| No Real Demand | 1,678 | 0.217 | `monitor_only` |
| Steady / No Flag | 17,062 | 0.511 | `monitor` |

Four of the five decline-risk archetypes cluster around a **0.69 observed decline rate** —
clearly above the 0.542 overall base rate — which is the honest, decision-support-level signal
behind prioritizing them. `Thin But Wanted` has n=1 (the rule will populate further as new
content gets scored); I'm keeping the rule but not reporting a rate off one row. `No Real
Demand` sits well *below* base rate (0.217) — pages with no real search demand can't "decline"
from a baseline they never had, which is a useful reason to leave them at low priority rather
than flag them.

**Why rank by model probability instead of the Week-4 baseline's raw-impressions rule:** the
model beat that rule baseline on precision@50 under the identical honest split (0.540 vs 0.240
— see `w05_model.ipynb`), so it's the better *ordering* signal. The reason codes and archetypes
above are what make that order explainable, not the probability number by itself.

**A caution on small-K noise, found while building this queue:** precision@50 (0.400) and
precision@100 (0.430) sit *below* the 0.542 base rate, while precision@200 (0.620), @500
(0.684), and @1000 (0.714) sit comfortably above it. With only 32 clients behind this data, the
very top of any single ranking is noisy. I would not hand someone a top-10 or top-50 list on its
own for this reason — the signal is real, but it shows up in batches of hundreds, not in the top
handful. This becomes a monitoring trigger in Section 4.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Setup — repo root + data (same as w05/w06)
import os
import json
import numpy as np
import pandas as pd

if not os.path.exists("data/raw/content_refresh_anonymized.csv"):
    if not os.path.exists("flyrank-ml-internship"):
        get_ipython().system('git clone https://github.com/TheAlishbahWaheed/flyrank-ml-internship.git')
    os.chdir("flyrank-ml-internship")

RANDOM_STATE = 42
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"].str.lower() == "down").astype(int)

# Identical leakage-safe feature set to Week 5 / Week 6 (scripts/ml_utils.py: MODEL_*_FEATURES)
df["log_impressions_90d"] = np.log1p(df["impressions_90d"])
df["log_clicks_90d"] = np.log1p(df["clicks_90d"])
df["log_sessions_90d"] = np.log1p(df["sessions_90d"])
df["log_ai_sessions_90d"] = np.log1p(df["ai_sessions_90d"])
df["has_keyword_data"] = df["search_volume"].notna().astype(int)
df["has_word_count"] = df["word_count"].notna().astype(int)
df["has_scroll_data"] = df["scroll_rate"].notna().astype(int)

NUM_FEATS = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d",
    "days_with_impressions", "days_with_sessions", "content_age_days", "days_since_last_update",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
    "has_keyword_data", "has_word_count", "has_scroll_data",
]
CAT_FEATS = [
    "competition_level", "content_type", "main_intent", "age_tier", "freshness_tier",
    "word_count_tier", "impression_tier", "position_tier",
]
for c in ["search_volume", "competition", "cpc", "word_count", "char_count", "scroll_rate"]:
    df[c] = df[c].fillna(0)
for c in CAT_FEATS:
    df[c] = df[c].fillna("unknown").astype(str)

X = df[NUM_FEATS + CAT_FEATS]
y = df["is_declining_label"].values
groups = df["client_id"].values

print("Shape:", df.shape, "| base decline rate:", round(y.mean(), 3))
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Shape: (30000, 52) | base decline rate: 0.542


In [3]:
# Re-fit the Week-5 model under the SAME honest, client-grouped split as w06
# (this is the audited number this playbook reports) -- then also produce 5-fold
# client-grouped out-of-fold (OOF) probabilities so every row in the full 30,000
# gets a genuine out-of-sample score to build the queue from.
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import roc_auc_score

def make_rf():
    pre = ColumnTransformer([
        ("num", "passthrough", NUM_FEATS),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_FEATS),
    ])
    return Pipeline([("pre", pre), ("clf", RandomForestClassifier(
        n_estimators=300, max_depth=8, min_samples_leaf=20,
        random_state=RANDOM_STATE, n_jobs=-1))])

def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))
    return float(np.asarray(y_true)[order[:k]].mean())

# 1) Single grouped split -- reproduces w06's reported, audited number
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(df, y, groups))
rf_single = make_rf()
rf_single.fit(X.iloc[train_idx], y[train_idx])
proba_test = rf_single.predict_proba(X.iloc[test_idx])[:, 1]
auc_single = roc_auc_score(y[test_idx], proba_test)
p50_single = precision_at_k(y[test_idx], proba_test, 50)
print(f"Reported (matches w06): grouped-split AUC {auc_single:.3f} | precision@50 {p50_single:.3f} "
      f"| test base rate {y[test_idx].mean():.3f}")

# 2) 5-fold client-grouped OOF -- covers the FULL population for this playbook's queue
gkf = GroupKFold(n_splits=5)
oof_proba = np.zeros(len(df))
fold_aucs = []
for tr_idx, te_idx in gkf.split(X, y, groups):
    rf = make_rf()
    rf.fit(X.iloc[tr_idx], y[tr_idx])
    p = rf.predict_proba(X.iloc[te_idx])[:, 1]
    oof_proba[te_idx] = p
    fold_aucs.append(roc_auc_score(y[te_idx], p))

df["oof_model_proba"] = oof_proba
oof_auc = roc_auc_score(y, oof_proba)
print(f"OOF (used to score full queue): AUC {oof_auc:.3f} | per-fold AUCs {[round(a,3) for a in fold_aucs]}")

for k in [50, 100, 200, 500, 1000]:
    print(f"Ranked-queue precision@{k}: {precision_at_k(y, oof_proba, k):.3f}")
print(f"Base rate (whole dataset): {y.mean():.3f}")

Reported (matches w06): grouped-split AUC 0.603 | precision@50 0.540 | test base rate 0.517
OOF (used to score full queue): AUC 0.673 | per-fold AUCs [np.float64(0.654), np.float64(0.602), np.float64(0.735), np.float64(0.647), np.float64(0.662)]
Ranked-queue precision@50: 0.400
Ranked-queue precision@100: 0.430
Ranked-queue precision@200: 0.620
Ranked-queue precision@500: 0.684
Ranked-queue precision@1000: 0.714
Base rate (whole dataset): 0.542


In [4]:
# Reason codes: transparent, threshold-based, human-readable
df["stale_and_visible"] = df["freshness_tier"].isin(["91-180", "181+"]) & \
    df["impression_tier"].isin(["moderate", "good", "excellent"])
df["high_demand_low_ctr"] = (df["impressions_90d"] >= 500) & (df["avg_position"] > 0) & \
    (df["avg_position"] <= 20) & (df["ctr"] < 0.5)
df["low_engagement"] = (df["sessions_90d"] >= 30) & (
    ((df["engagement_rate"] > 0) & (df["engagement_rate"] < 30)) |
    ((df["scroll_rate"] > 0) & (df["scroll_rate"] < 30))
)
df["thin_but_visible"] = df["word_count_tier"].eq("<1000") & \
    df["impression_tier"].isin(["moderate", "good", "excellent"])
df["already_winning"] = df["position_tier"].isin(["top_3", "page_1"]) & (df["ctr"] >= 0.5)
df["no_real_demand"] = (df["impression_tier"] == "low") & (df["has_keyword_data"] == 0)
df["model_decline_risk"] = df["oof_model_proba"] >= 0.65

def reason_codes(row):
    codes = []
    if row["model_decline_risk"]: codes.append("model_decline_risk")
    if row["stale_and_visible"]: codes.append("stale_and_visible")
    if row["high_demand_low_ctr"]: codes.append("high_demand_low_ctr")
    if row["low_engagement"]: codes.append("low_engagement")
    if row["thin_but_visible"]: codes.append("thin_but_visible")
    if row["already_winning"]: codes.append("already_winning")
    if row["no_real_demand"]: codes.append("no_real_demand")
    return "|".join(codes) if codes else "no_flag"

df["reason_code"] = df.apply(reason_codes, axis=1)

# Archetypes: named groups built from the SAME flags above (priority order, first match wins).
# Rule-based cross, not a statistical cluster.
def archetype(row):
    if row["already_winning"]:
        return "Champion (protect)"
    if row["thin_but_visible"] and row["model_decline_risk"]:
        return "Thin But Wanted"
    if row["stale_and_visible"] and row["model_decline_risk"]:
        return "Stale Workhorse"
    if row["high_demand_low_ctr"] and row["model_decline_risk"]:
        return "Visible But Under-Clicked"
    if row["low_engagement"] and row["model_decline_risk"]:
        return "Disengaging Reader"
    if row["model_decline_risk"]:
        return "General Decline Risk"
    if row["no_real_demand"]:
        return "No Real Demand"
    return "Steady / No Flag"

df["archetype"] = df.apply(archetype, axis=1)

ARCHETYPE_ACTION = {
    "Champion (protect)": "protect_do_not_touch",
    "Thin But Wanted": "expand_and_refresh",
    "Stale Workhorse": "refresh_priority",
    "Visible But Under-Clicked": "rewrite_title_and_meta",
    "Disengaging Reader": "review_engagement_and_layout",
    "General Decline Risk": "refresh_review",
    "No Real Demand": "monitor_only",
    "Steady / No Flag": "monitor",
}
df["action"] = df["archetype"].map(ARCHETYPE_ACTION)

def confidence(row):
    if row["model_decline_risk"] and row["impressions_90d"] >= 500 and row["sessions_90d"] >= 10:
        return "high"
    if row["oof_model_proba"] >= 0.5:
        return "medium"
    return "low"

df["confidence"] = df.apply(confidence, axis=1)

review = (df.groupby("archetype")
          .agg(n=("content_id", "size"), decline_rate=("is_declining_label", "mean"),
               avg_oof_proba=("oof_model_proba", "mean"))
          .sort_values("n", ascending=False))
print(review.round(3))
print()
print("Action counts:\n", df["action"].value_counts())
print()
print("Confidence counts:\n", df["confidence"].value_counts())

                               n  decline_rate  avg_oof_proba
archetype                                                    
Steady / No Flag           17062         0.511          0.507
General Decline Risk        3555         0.692          0.700
Champion (protect)          2647         0.471          0.541
Visible But Under-Clicked   2582         0.686          0.696
Stale Workhorse             2254         0.687          0.702
No Real Demand              1678         0.217          0.295
Disengaging Reader           221         0.692          0.701
Thin But Wanted                1         1.000          0.698

Action counts:
 action
monitor                         17062
refresh_review                   3555
protect_do_not_touch             2647
rewrite_title_and_meta           2582
refresh_priority                 2254
monitor_only                     1678
review_engagement_and_layout      221
expand_and_refresh                  1
Name: count, dtype: int64

Confidence counts:
 confi

In [5]:
# Top of the ranked queue (rank 1-10), for a human to sanity-check by eye
ranked_preview = df.sort_values("oof_model_proba", ascending=False).reset_index(drop=True)
ranked_preview["rank"] = np.arange(1, len(ranked_preview) + 1)
preview_cols = ["rank", "content_id", "oof_model_proba", "confidence", "archetype", "action",
                "reason_code", "impressions_90d", "avg_position", "ctr", "is_declining_label"]
print(ranked_preview.head(10)[preview_cols].to_string(index=False))

 rank           content_id  oof_model_proba confidence            archetype           action                                              reason_code  impressions_90d  avg_position  ctr  is_declining_label
    1 content_a5a2fbc76336         0.846179     medium      Stale Workhorse refresh_priority                     model_decline_risk|stale_and_visible              307          39.8  0.0                   0
    2 content_2ba626fea4d6         0.844780     medium      Stale Workhorse refresh_priority                     model_decline_risk|stale_and_visible              360           7.2  0.0                   0
    3 content_e988c1699454         0.843416     medium      Stale Workhorse refresh_priority                     model_decline_risk|stale_and_visible             2197          21.5  0.0                   1
    4 content_a662ef2af9b4         0.840366     medium      Stale Workhorse refresh_priority model_decline_risk|stale_and_visible|high_demand_low_ctr              542          

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

## Intended use

This playbook is decision-support for a content or SEO team deciding **which existing pages to
review first**, using patterns observed in one bundled, anonymized 30,000-row / 32-client
starter dataset covering a trailing 90-day window. It ranks and groups; it does not publish,
rewrite, or take any action by itself.

Appropriate uses:
- Ordering a human reviewer's queue for content refresh work.
- Giving a reviewer a starting reason (`reason_code`, `archetype`) instead of a bare number.
- Flagging pages worth a second look for a CTR or engagement issue — not diagnosing the cause.

## Limits

- **Modest, not strong, model skill.** ROC-AUC of 0.603 on the honest, client-grouped split
  means better than chance (0.5) but well short of a strong classifier. Precision@50 (0.540) is
  barely above that split's own base rate (0.517) — most of the model's value shows up in
  batches of hundreds, not the very top few picks (Section 1's precision@K note).
- **Cross-client generalization is the weak point.** `w06_validation_audit.ipynb`'s own
  before/after check found ROC-AUC dropped from 0.752 (naive random split, clients leak across
  train/test) to 0.603 (grouped split, unseen clients) — roughly a third of the naive split's
  apparent skill was the model recognizing *which client* a row belonged to, not predicting
  decline. This model has not been shown to generalize to a brand-new client with no training
  history.
- **One dataset, one 90-day window, 32 clients.** `content_type` is 91% "keyword article"; the
  other two types are thin, and any archetype conclusion for them individually is weak.
- **Cross-sectional, not causal.** This is one snapshot with an observed association between
  staleness/visibility/engagement and decline. It does **not** show that refreshing a page
  *causes* it to recover — no experiment (A/B test, before/after refresh comparison) sits behind
  this model.
- **The label is a rule, not ground truth.** `is_declining_label` comes from a
  30-day-vs-prior-30-day impression-trend threshold — transparent and reasonable, but a
  definitional choice, not an objective fact about a page's health.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Numbers backing the limits above
content_type_share = df["content_type"].value_counts(normalize=True).round(3)
print("content_type share (limit: one type dominates):\n", content_type_share)
print()
print(f"Grouped-split AUC (audited, reported): {auc_single:.3f}")
print(f"OOF AUC used for the full queue:       {oof_auc:.3f}")
print(f"Per-fold AUC spread:                   {min(fold_aucs):.3f} - {max(fold_aucs):.3f} "
      f"(range {max(fold_aucs) - min(fold_aucs):.3f}, across 32 clients)")
print()
print("Top model features (from w05_model.ipynb's Random Forest, for reference):")
rf_features = rf_single.named_steps["pre"].named_transformers_["cat"].get_feature_names_out(CAT_FEATS)
all_feature_names = NUM_FEATS + list(rf_features)
importances = pd.Series(rf_single.named_steps["clf"].feature_importances_, index=all_feature_names)
print(importances.sort_values(ascending=False).head(8).round(3))
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


content_type share (limit: one type dominates):
 content_type
keyword article       0.907
feedly article        0.070
comparison article    0.023
Name: proportion, dtype: float64

Grouped-split AUC (audited, reported): 0.603
OOF AUC used for the full queue:       0.673
Per-fold AUC spread:                   0.602 - 0.735 (range 0.134, across 32 clients)

Top model features (from w05_model.ipynb's Random Forest, for reference):
days_with_impressions    0.154
log_impressions_90d      0.139
avg_position             0.105
content_age_days         0.089
char_count               0.043
word_count               0.041
position_tier_top_3      0.033
log_clicks_90d           0.029
dtype: float64


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

## What a human must check before acting

For every item pulled from this queue, before any change is made:
1. **Confirm the reason code against the live page**, not just the row. `high_demand_low_ctr`
   in the data could be a real CTR problem, or a mis-tagged page, a redirect, a duplicate, or a
   page that recently changed search intent.
2. **Check `confidence` first.** High confidence requires both a flagged decline risk
   (probability ≥ 0.65) *and* real traffic (≥500 impressions, ≥10 sessions) — 3,176 of 30,000
   rows (10.6%) qualify (numbers below). Medium (18,228 rows, 60.8%) and low (8,596 rows, 28.7%)
   confidence items need more scrutiny before any resourcing decision, and low-confidence items
   should not be the sole basis for an action.
3. **Read the editorial and brand context the model never sees** — legal/compliance pages,
   seasonal content that's supposed to look "stale" outside its season, recently-published pages
   still ramping up, pages mid-refresh already.
4. **Rewrite for a human, not a scoreboard.** The staleness/visibility association is
   directional, from one snapshot — not a guarantee any specific page will recover if touched.

## Should never be automated

- **Auto-publishing any content change** (title, meta, body rewrite) from the score or
  archetype alone — every `rewrite_title_and_meta`, `refresh_priority`, or `expand_and_refresh`
  action needs a human editor in the loop.
- **Auto-pruning or unpublishing** `Steady / No Flag` or `No Real Demand` pages — low predicted
  risk isn't the same as "worthless"; some are stable performers or brand-required pages.
- **Using this score to evaluate an individual writer's or team's performance.** The label
  reflects page-level metric movement, not the quality of anyone's work, and the model's own
  cross-client weakness (0.603 AUC) means it isn't reliable enough to attach to a person's
  review.
- **Treating `Champion (protect)` as "never touch."** It means "don't prioritize for refresh
  right now" — a real, human-flagged issue (e.g. a factual error) still overrides the archetype.
- **Trusting the score at face value for a brand-new client** with no training history — see
  the generalization limit in Section 2; a new client's first flagged batch should get a wider
  human-review pass than a routine one.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Numbers backing the human-review section
conf_counts = df["confidence"].value_counts()
conf_share = df["confidence"].value_counts(normalize=True).round(3)
print("Confidence counts:\n", conf_counts)
print("\nConfidence share:\n", conf_share)
print()
print("Protect / no-go group size (Champion, archetype):",
      int((df["archetype"] == "Champion (protect)").sum()))
print("Low-confidence rows (should not drive a standalone action):",
      int((df["confidence"] == "low").sum()))
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Confidence counts:
 confidence
medium    18228
low        8596
high       3176
Name: count, dtype: int64

Confidence share:
 confidence
medium    0.608
low       0.287
high      0.106
Name: proportion, dtype: float64

Protect / no-go group size (Champion, archetype): 2647
Low-confidence rows (should not drive a standalone action): 8596


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

## What would tell us this has gone stale

**Retrain triggers (data/behavior drift):**
- A new reporting window becomes available (next refresh of this dataset, or the live
  warehouse) — retrain rather than reuse these weights past roughly one quarter, since the
  label itself (a 30-day trend) is a moving target.
- A meaningful shift in the features the model relies on most — `days_with_impressions`,
  `log_impressions_90d`, and `avg_position` are the top drivers (Section 2) — for example a big
  swing in average `content_age_days` or the `word_count_tier` mix signals the population has
  changed since training.
- A wave of brand-new clients with no training history — exactly the case the model is weakest
  on (cross-client AUC 0.603) — is a retrain/re-audit trigger on its own, not just a monitoring
  note.

**Monitoring triggers (queue health — watch before assuming a retrain is needed):**
- **The fold-to-fold AUC spread widens further.** This run's 5-fold OOF spread was 0.602–0.735
  (range 0.133, printed in Section 2). A much wider spread on a future re-run would suggest the
  client population is getting more heterogeneous than the model can track.
- **Batch precision@K drifts toward the base rate.** Precision@200 here is 0.620 against a 0.542
  base rate — a real but modest lift. If live tracking (comparing a quarter's flagged batch
  against realized outcomes) shows that lift shrinking toward zero, that's the signal to stop
  trusting the ranking — not just recomputing the historical number.
- **Archetype sizes shift sharply** — e.g. `Champion (protect)`'s share collapsing, or
  `No Real Demand` swelling — could mean a real change in the content portfolio or a broken
  upstream feature; check both before acting on it.
- **Reviewer disagreement rate rises.** If human reviewers are overriding or rejecting a growing
  share of `high`-confidence flags, treat that as an earlier warning than any recomputed metric.

No formal SLA cadence is set here — "at least once a quarter, or with the next data refresh,
whichever comes first" is a light, honest starting point for a decision-support tool, not a
production system.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Baseline numbers this section's triggers are measured against, so a future re-run
# has something concrete to compare to.
precision_curve = {k: round(precision_at_k(y, oof_proba, k), 3) for k in [50, 100, 200, 500, 1000]}
monitoring_baseline = {
    "fold_auc_min": round(min(fold_aucs), 3),
    "fold_auc_max": round(max(fold_aucs), 3),
    "fold_auc_range": round(max(fold_aucs) - min(fold_aucs), 3),
    "precision_at_k": precision_curve,
    "base_rate": round(float(y.mean()), 3),
    "archetype_share": (df["archetype"].value_counts(normalize=True)).round(3).to_dict(),
}
print(json.dumps(monitoring_baseline, indent=2))
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


{
  "fold_auc_min": 0.602,
  "fold_auc_max": 0.735,
  "fold_auc_range": 0.134,
  "precision_at_k": {
    "50": 0.4,
    "100": 0.43,
    "200": 0.62,
    "500": 0.684,
    "1000": 0.714
  },
  "base_rate": 0.542,
  "archetype_share": {
    "Steady / No Flag": 0.569,
    "General Decline Risk": 0.118,
    "Champion (protect)": 0.088,
    "Visible But Under-Clicked": 0.086,
    "Stale Workhorse": 0.075,
    "No Real Demand": 0.056,
    "Disengaging Reader": 0.007,
    "Thin But Wanted": 0.0
  }
}


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

## What gets exported

- `work/outputs/action_playbook_queue.csv` — the full 30,000-row ranked queue (rank, score,
  confidence, archetype, action, reason_code, and the supporting metrics a reviewer needs).
  Regenerated by the cell below, not committed — the repo's leak-guard blocks data files in CI,
  and this notebook is the source of truth for rebuilding it.
- `work/outputs/action_playbook_summary.json` — the headline numbers (AUC, precision@K,
  archetype/action/confidence counts) this notebook's claims trace back to.
- `work/figures/archetype_counts.png`, `archetype_decline_rate.png`, `precision_at_k.png` — the
  three figures backing Sections 1 and 4, committed to the repo for reuse in the paper.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Export: ranked queue CSV + summary JSON + figures
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/figures", exist_ok=True)

ranked = df.sort_values("oof_model_proba", ascending=False).reset_index(drop=True)
ranked["rank"] = np.arange(1, len(ranked) + 1)
export_cols = ["rank", "content_id", "client_id", "oof_model_proba", "confidence",
               "archetype", "action", "reason_code",
               "impressions_90d", "sessions_90d", "avg_position", "ctr",
               "freshness_tier", "impression_tier", "word_count_tier", "content_type"]
queue_path = "work/outputs/action_playbook_queue.csv"
ranked[export_cols].to_csv(queue_path, index=False)
print("Wrote:", queue_path, "-", len(ranked), "rows")

actionable = df[~df["action"].isin(["monitor", "monitor_only", "protect_do_not_touch"])]
summary = {
    "generated_by": "w07_action_playbook.ipynb",
    "rows_scored": int(len(df)),
    "validated_single_split_grouped_auc": round(auc_single, 3),
    "validated_single_split_grouped_p50": round(p50_single, 3),
    "oof_5fold_grouped_auc": round(oof_auc, 3),
    "oof_fold_aucs": [round(a, 3) for a in fold_aucs],
    "base_rate": round(float(y.mean()), 3),
    "archetype_counts": df["archetype"].value_counts().to_dict(),
    "action_counts": df["action"].value_counts().to_dict(),
    "confidence_counts": df["confidence"].value_counts().to_dict(),
    "actionable_queue_size": int(len(actionable)),
}
with open("work/outputs/action_playbook_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print("Wrote: work/outputs/action_playbook_summary.json")

# Figure 1: archetype counts
fig, ax = plt.subplots(figsize=(8, 4.5))
order = df["archetype"].value_counts()
ax.barh(order.index[::-1], order.values[::-1], color="#426B69")
ax.set_xlabel("Number of content items")
ax.set_title("Content items by archetype")
plt.tight_layout()
plt.savefig("work/figures/archetype_counts.png", dpi=150)
plt.close()

# Figure 2: decline rate by archetype vs base rate
fig, ax = plt.subplots(figsize=(8, 4.5))
rates = df.groupby("archetype")["is_declining_label"].mean().sort_values()
ax.barh(rates.index, rates.values, color="#8C6BB1")
ax.axvline(df["is_declining_label"].mean(), color="black", linestyle="--", linewidth=1,
           label="Overall base rate")
ax.set_xlabel("Observed decline rate (audit label, not a score input)")
ax.set_title("Decline rate by archetype vs. overall base rate")
ax.legend()
plt.tight_layout()
plt.savefig("work/figures/archetype_decline_rate.png", dpi=150)
plt.close()

# Figure 3: precision@K of the ranked queue vs base rate
fig, ax = plt.subplots(figsize=(7, 4.5))
ks = [50, 100, 200, 500, 1000]
precs = [precision_at_k(y, oof_proba, k) for k in ks]
ax.plot([str(k) for k in ks], precs, marker="o", color="#4E79A7", label="Ranked-queue precision@K")
ax.axhline(y.mean(), color="black", linestyle="--", linewidth=1, label="Base rate")
ax.set_ylabel("Precision (share actually declining)")
ax.set_xlabel("K (top-K of the ranked queue)")
ax.set_title("Ranked-queue precision@K vs. base rate")
ax.legend()
plt.tight_layout()
plt.savefig("work/figures/precision_at_k.png", dpi=150)
plt.close()

print("Wrote 3 figures to work/figures/")
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Wrote: work/outputs/action_playbook_queue.csv - 30000 rows
Wrote: work/outputs/action_playbook_summary.json
Wrote 3 figures to work/figures/


## Self-check

Before you submit, confirm each line honestly:

- [ ✅] Every section above is filled — markdown thinking AND the code that backs it
- [✅ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅ ] No client names, URLs, or private queries anywhere
- [ ✅] My claims use careful words: observed, measured, directional, decision-support
- [✅ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.